In [103]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error
import time
#import pygwalker as pyg
from datetime import datetime
import os
#from ydata_profiling import ProfileReport
import csv
import seaborn as sns

In [104]:
# train_df = pd.read_csv('../artifacts/train.csv')
# test_df = pd.read_csv('../artifacts/test.csv')
df = pd.read_csv('../artifacts/raw.csv')

## Imputing missing values ##

In [105]:
df['CompetitionDistance'].fillna(0,inplace=True)
#train_df.isnull().sum()

In [106]:
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

df['CompetitionOpen_missing'] = df['CompetitionOpenSinceYear'].isna().astype(int)
df['CompetitionOpenSinceMonth'].fillna(1,inplace=True)
df['CompetitionOpenSinceYear'].fillna(df['Year'].min(),inplace=True)
df['CompetitionOpen'] = 12 * (df['Year'] - df['CompetitionOpenSinceYear']) + (df['Month'] - 
                                                                              df['CompetitionOpenSinceMonth'])
df['CompetitionOpen'] = df['CompetitionOpen'].apply(lambda x: max(x, 0))


In [107]:
df['WeekOfYear'] = df['Date'].dt.isocalendar().week
df['Promo2OpenSinceMonths'] = 12 * (df['Year'] - df['Promo2SinceYear']) + (df['WeekOfYear'] - 
                                                                           df['Promo2SinceWeek']) / 4.0
df['Promo2OpenSinceMonths'] = df['Promo2OpenSinceMonths'].apply(lambda x: max(x, 0) if pd.notnull(x) else 0)
df.loc[df['Promo2'] == 0, 'Promo2OpenSinceMonths'] = 0
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
df['MonthStr'] = df['Month'].map(month_map)
promo_months = df['PromoInterval'].fillna('').str.split(',')
df['IsPromoMonth'] = [
    1 if m in months else 0 
    for m, months in zip(df['MonthStr'], promo_months)
]


## Feature Engineering ##

In [108]:
df['StateHoliday'] = np.where((df['StateHoliday'] == '0') | (df['StateHoliday'] == 0),0,1)

In [109]:
df['Assortment'] = np.where(df['Assortment'] == 'b','b','other')

In [110]:
df.sample(5)

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,PromoInterval,Year,Month,Day,CompetitionOpen_missing,CompetitionOpen,WeekOfYear,Promo2OpenSinceMonths,MonthStr,IsPromoMonth
755255,71,5,2013-08-23,5779,593,1,0,0,1,a,...,"Mar,Jun,Sept,Dec",2013,8,23,0,60.0,34,47.25,Aug,0
856686,37,5,2013-05-24,6434,823,1,0,0,0,c,...,NaN,2013,5,24,0,0.0,21,0.00,May,0
762359,485,6,2013-08-17,3733,411,1,0,0,0,d,...,"Jan,Apr,Jul,Oct",2013,8,17,1,7.0,33,14.75,Aug,0
709394,1040,5,2013-10-04,9430,1014,1,0,0,0,a,...,"Jan,Apr,Jul,Oct",2013,10,4,0,8.0,40,0.00,Oct,1
444744,645,4,2014-05-29,0,0,0,0,1,0,a,...,"Feb,May,Aug,Nov",2014,5,29,1,16.0,22,54.25,May,1


In [111]:
df = df[df['Open'] == 1].copy()

In [112]:
df = df[df['Date'] != pd.Timestamp('2015-07-04')]

In [113]:
df.shape

(843278, 27)

In [114]:
df = df.sort_values('Date')

cutoff_date = '2015-06-01'

train = df[df['Date'] < cutoff_date]
valid = df[df['Date'] >= cutoff_date]

store_avg_sales = train.groupby('Store')['Sales'].mean().rename('Store_avg_sales')
store_avg_customers = train.groupby('Store')['Customers'].mean().rename('Store_avg_customers')
train = train.merge(store_avg_sales, on='Store', how='left')
train = train.merge(store_avg_customers, on='Store', how='left')
valid = valid.merge(store_avg_sales, on='Store', how='left')
valid = valid.merge(store_avg_customers, on='Store', how='left')

num_col=['CompetitionDistance','CompetitionOpen','Promo2OpenSinceMonths','Store_avg_sales','Store_avg_customers']
cat_col = ['Year','StoreType','Assortment']

X_train = train.drop(['Sales', 'Date'], axis=1)
y_train = train['Sales']

X_test = valid.drop(['Sales', 'Date'], axis=1)
y_test = valid['Sales']

In [115]:
X_train.drop(['Customers','CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear','Promo2SinceWeek','Promo2SinceYear','PromoInterval','MonthStr'], axis=1, inplace=True)
X_test.drop(['Customers','CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear','Promo2SinceWeek','Promo2SinceYear','PromoInterval','MonthStr'], axis=1, inplace=True)

In [116]:
preprocessor = ColumnTransformer([
    ('scl',StandardScaler(),num_col),
    ('ohe',OneHotEncoder(drop='first',sparse_output=True),cat_col)
],remainder='passthrough')

In [117]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [118]:
y_train_log = np.log1p(y_train)

In [119]:
results_file = 'rmsep_score.csv'
file_exists = os.path.isfile(results_file)

models = {
    "XGBRegressor" : XGBRegressor(tree_method='hist',n_jobs=-1,random_state=42),
    'LinearRegression': LinearRegression(),
    'LGBMRegressor': LGBMRegressor(n_jobs=-1)
}

results = []

for model_name,model in models.items():
    y_test_mean = np.mean(y_test)

    training_start = time.perf_counter()
    model.fit(X_train,y_train_log)
    training_stop = time.perf_counter()
    training_time_taken = training_stop - training_start

    prediction_start = time.perf_counter()
    pred_log = model.predict(X_test)
    prediction_stop = time.perf_counter()

    Prediction = np.expm1(pred_log)
    #prediction = pred_log

    prediction_time_taken = prediction_stop-prediction_start
    # rmse = root_mean_squared_error(y_test,prediction)
    # rmsep = rmse/y_test_mean

    def rmspe(y_true, y_pred):
        mask = y_true != 0
        return np.sqrt(np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2))

    score = rmspe(y_test.values, Prediction)


    print(f'{model_name}: {score*100:.2f}%\n')
    print(f'Training time taken for {model_name}: {training_time_taken:.4f}\n')
    print(f'prediction time taken for {model_name}: {prediction_time_taken:.4f}\n')

    # importance = pd.Series(model.feature_importances_, index=preprocessor.get_feature_names_out())
    # print(importance.sort_values(ascending=False).head(25))

    results.append({
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'model_name': model_name,
        #'rmse': rmse,
        'rmsep_percent': score * 100,
        'prediction_time_taken_sec': prediction_time_taken,
        'training_time_taken_sec': training_time_taken
    })

    errors = np.abs((y_test.values - Prediction) / y_test.values)
    error_df = valid.copy()
    error_df['prediction'] = Prediction
    error_df['pct_error'] = errors

    print(error_df.sort_values('pct_error', ascending=False).head(30)['Store'].value_counts())
    print(error_df.sort_values('pct_error', ascending=False).head(30)['Date'].value_counts())


    print(error_df.sort_values('pct_error', ascending=False).head(30)[
        ['Store','Date','Sales','prediction','pct_error','Promo','StateHoliday','StoreType','DayOfWeek']
    ])

with open(results_file, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['timestamp', 'model_name', 'rmsep_percent', 'prediction_time_taken_sec','training_time_taken_sec'])
    if not file_exists:
        writer.writeheader()
    writer.writerows(results)



XGBRegressor: 15.63%

Training time taken for XGBRegressor: 6.5504

prediction time taken for XGBRegressor: 0.1136

Store
415     5
271     4
286     3
909     2
782     2
749     2
292     2
559     2
483     2
876     1
917     1
501     1
1014    1
956     1
534     1
Name: count, dtype: int64
Date
2015-07-25    6
2015-06-06    3
2015-06-13    3
2015-06-27    3
2015-07-01    1
2015-07-10    1
2015-06-26    1
2015-06-01    1
2015-06-04    1
2015-06-05    1
2015-06-03    1
2015-07-15    1
2015-06-29    1
2015-07-18    1
2015-07-08    1
2015-06-08    1
2015-07-11    1
2015-07-02    1
2015-06-20    1
Name: count, dtype: int64
       Store       Date  Sales    prediction  pct_error  Promo  StateHoliday  \
36352    292 2015-07-10   1012   5595.406250   4.529058      0             0   
24938    782 2015-06-26   1422   5470.867188   2.847305      0             0   
29038    909 2015-07-01   3547  13621.378906   2.840253      1             0   
2262     415 2015-06-03   2238   7361.745605   